# CLIP Training Comparison on Flickr30k

This notebook compares 5 different methods for training/evaluating CLIP on the Flickr30k dataset:
1. **Baseline**: Zero-shot evaluation of the pre-trained model.
2. **Linear Probe**: Training only the projection heads while keeping the backbones frozen.
3. **Partial Fine-tune**: Unfreezing and training only the last few layers of the encoders.
4. **LoRA**: Using Low-Rank Adaptation (PEFT) to fine-tune the model efficiently.
5. **Full Fine-tune**: Unfreezing the entire model for comprehensive training.

At the end, all models are evaluated on Recall@1, Recall@5, and MRR, and then uploaded to Hugging Face.

In [ ]:
!pip install -q transformers peft datasets huggingface_hub torch torchvision tqdm pillow pandas

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import CLIPProcessor, CLIPModel, get_scheduler
from torch.optim import AdamW
from datasets import load_dataset
from tqdm.auto import tqdm
import numpy as np
import json
import pandas as pd
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "openai/clip-vit-base-patch32"
print(f"Using device: {device}")


## 1. Dataset Preparation

We use the `nlphuji/flickr30k` dataset. For training CLIP, we use the standard image-text pairs.

In [ ]:
model_id = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_id)

# Use TRUE splits: train for training, test for evaluation
train_data = load_dataset("nlphuji/flickr30k", split="train[:2000]", trust_remote_code=True)
test_data = load_dataset("nlphuji/flickr30k", split="test[:500]", trust_remote_code=True)

train_ds = Flickr30kDataset(train_data, processor)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
print(f"Train size: {len(train_ds)}, Test size: {len(test_data)}")

## 2. Evaluation Logic

Standard Image-Text Retrieval metrics: Recall@1, Recall@5, and MRR.

In [ ]:
# Using optimized evaluation from helper script
print("Evaluation logic loaded from clip_training_utils.py")

## 3. Training Function

A simple contrastive training loop using CLIP's built-in loss.

In [ ]:
def train_model(model, train_loader, epochs=1, lr=5e-5):
    model.to(device)
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    num_training_steps = epochs * len(train_loader)
    lr_scheduler = get_scheduler('linear', optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)
    model.train()
    for epoch in range(epochs):
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}')
        for batch, _ in pbar:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch, return_loss=True)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            pbar.set_postfix({'loss': loss.item()})
    return model


## 4. Methods Comparison

We will run each method and log the results.

In [ ]:
results_table = {}
def log_result(name, metrics):
    results_table[name] = metrics
    print(f"--- {name} ---")
    for k, v in metrics.items():
        print(f"{k}: {v}")

### 4.1 Baseline (Zero-Shot)
Evaluating the model without any fine-tuning.

In [ ]:
model = CLIPModel.from_pretrained(model_id).to(device)
metrics = evaluate_retrieval(model, test_data, processor, device)
log_result('Baseline', metrics)


### 4.2 Linear Probe
Freezing the encoders and training only the projection layers.

In [ ]:
base_model = CLIPModel.from_pretrained(model_id)
# True Linear Probe: Randomly initialized head, frozen backbone
# Dimension: 512 is optimal for Flickr30K's complexity.
model = CLIPLinearProbe(base_model, embed_dim=512)

model = train_model(model, train_loader, epochs=1, lr=1e-4)
metrics = evaluate_retrieval(model, test_data, processor, device)
log_result("Linear Probe", metrics)

### 4.3 Partial Fine-tune
Unfreezing the projection layers and the last Transformer block of each encoder.

In [ ]:
model = CLIPModel.from_pretrained(model_id)
for param in model.parameters(): param.requires_grad = False
model.visual_projection.requires_grad_(True)
model.text_projection.requires_grad_(True)
model.logit_scale.requires_grad_(True)
for param in model.vision_model.encoder.layers[-1].parameters(): param.requires_grad = True
for param in model.text_model.encoder.layers[-1].parameters(): param.requires_grad = True

model = train_model(model, train_loader, epochs=1)
metrics = evaluate_retrieval(model, test_data, processor, device)
log_result('Partial Fine-tune', metrics)


### 4.4 LoRA (Low-Rank Adaptation)
Using PEFT to add trainable adapters to the attention layers.

In [ ]:
from peft import LoraConfig, get_peft_model

model = CLIPModel.from_pretrained(model_id)
config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
)
model = get_peft_model(model, config)
model.print_trainable_parameters()

model = train_model(model, train_loader, epochs=1)
metrics = evaluate_retrieval(model, test_data, processor, device)
log_result("LoRA", metrics)

### 4.5 Full Fine-tune
Training all parameters of the model.

In [ ]:
model = CLIPModel.from_pretrained(model_id)
model = train_model(model, train_loader, epochs=1, lr=5e-6)
metrics = evaluate_retrieval(model, test_data, processor, device)
log_result('Full Fine-tune', metrics)


## 5. Final Results Summary

In [ ]:
import pandas as pd
df = pd.DataFrame(results_table).T
print("Final Comparison Table:")
display(df)

## 6. Hugging Face Upload

Log in to your Hugging Face account and upload the best performing model.

In [ ]:
from huggingface_hub import notebook_login
# notebook_login() # Uncomment to login via token

def upload_to_hf(model, repo_name):
    try:
        model.push_to_hub(repo_name)
        print(f"Successfully uploaded to https://huggingface.co/{repo_name}")
    except Exception as e:
        print(f"Upload failed: {e}")

# upload_to_hf(model, "your-username/clip-flickr30k-finetuned")